# 🎯 Module 13 · 03 — Text Representation

> **Stage:** Machine Learning → NLP  
> **Level:** Beginner → Intermediate  
> **Module:** 13 — NLP Basics / 03 — Text Representation  
> **Date:** 27 January 2026  
> **Previous:** 02 — Text Preprocessing  
> **Next:** 04 — Bag of Words

---

After preprocessing, we have clean text — but ML models still can't work with sentences like:

```text
"I love this laptop"
```

**Text representation** converts text into **numerical features** a model can process.

In this notebook we'll cover:

- Why we need text representation
- The main representation families
- **One-Hot Encoding**
- **Sparse** representation
- **Bag of Words** (overview)
- **N-grams**
- **TF-IDF** (overview)
- **Word embeddings** (overview)
- Sparse vs Dense
- Document representation
- Text similarity
- Choosing the right representation
- Classical limitations

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain why text must be converted into numbers
- Build a **one-hot** vector for a word
- Explain **sparse** vs **dense** representations
- Describe what **BoW**, **N-grams**, and **TF-IDF** capture
- Explain what **word embeddings** are and why they're dense
- Compute **cosine similarity** between text vectors
- Choose an appropriate representation for a task
- Explain the **limitations** of classical representations

---
## ⚙️ 0. Setup & Imports

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('✅ Setup ready.')

In [ ]:
# A tiny corpus to reuse throughout the notebook
corpus = [
    "I love Python",
    "I love machine learning",
    "I hate this software",
    "machine learning tutorial",
    "football match today",
]

for i, doc in enumerate(corpus, 1):
    print(f'Doc {i}: {doc}')

---
## 🧠 1. Why Do We Need Text Representation?

Consider:

```text
"I love Python"
"I love machine learning"
"I hate this software"
```

A model cannot directly compute mathematical relationships between **strings**.

We need:

```text
Text
 ↓
Numerical Vector
 ↓
ML Model
```

> This is one of the **most important steps** in classical NLP.

---
## 🗂️ 2. Main Types of Text Representation

Classical NLP commonly uses:

```text
1. One-Hot Encoding
2. Bag of Words
3. N-Grams
4. TF-IDF
5. Word Embeddings
```

Broadly divided into:

| Type | Representation |
| --- | --- |
| **Sparse** | One-Hot, BoW, TF-IDF |
| **Dense** | Word Embeddings |

---
## 1️⃣ 3. One-Hot Encoding

Each word gets a vector where **one** position is `1` and **all others** are `0`.

Vocabulary:

```text
["cat", "dog", "car"]
```

Representations:

```text
cat → [1, 0, 0]
dog → [0, 1, 0]
car → [0, 0, 1]
```

### ⚠️ Problems

- Very **large** vectors with big vocabularies
- **No semantic relationships** — `king` and `queen` are as unrelated as `king` and `pizza`

In [ ]:
# Manual one-hot encoding
vocab = ['cat', 'dog', 'car']
word_to_idx = {w: i for i, w in enumerate(vocab)}

def one_hot(word, vocab):
    vec = [0] * len(vocab)
    if word in vocab:
        vec[vocab.index(word)] = 1
    return vec

print('Vocabulary:', vocab)
for w in vocab + ['unknown']:
    print(f'{w:<10} → {one_hot(w, vocab)}')

---
## 🕳️ 4. Sparse Representation

A vector is **sparse** when **most values are zero**:

```text
[0, 0, 0, 1, 0, 0, 0, 0, 1, 0]
```

Classical NLP methods (BoW, TF-IDF) usually produce **sparse** matrices.

### Why it matters

Real datasets produce matrices of size:

```text
Documents × Vocabulary
```

With vocabularies of 50k+ words and thousands of documents, most positions are zero.

Scikit-learn and friends use **sparse matrix formats** to save memory and compute.

In [ ]:
# Show sparsity of a tiny bag-of-words matrix
def build_vocab(corpus):
    vocab = set()
    for doc in corpus:
        vocab.update(doc.lower().split())
    return sorted(vocab)

def bow(doc, vocab):
    tokens = doc.lower().split()
    return [tokens.count(w) for w in vocab]

vocab = build_vocab(corpus)
matrix = np.array([bow(d, vocab) for d in corpus])

print('Vocabulary size:', len(vocab))
print('Matrix shape   :', matrix.shape)

sparsity = (matrix == 0).sum() / matrix.size
print(f'Sparsity       : {sparsity:.1%} of entries are zero')

df = pd.DataFrame(matrix, columns=vocab, index=[f'Doc{i+1}' for i in range(len(corpus))])
print('\nBag-of-Words matrix:')
print(df)

---
## 👜 5. Bag of Words (Overview)

**BoW** represents a document by **which words it contains and how often**.

### Example

```text
Document 1:  "I love Python"
Document 2:  "I love machine learning"
```

Vocabulary:

```text
["I", "love", "Python", "machine", "learning"]
```

Representations:

```text
Doc 1 → [1, 1, 1, 0, 0]
Doc 2 → [1, 1, 0, 1, 1]
```

### ⚠️ Key limitation

Word **order is ignored**.

```text
"dog bites man"
"man bites dog"
```

produce essentially the same representation.

> We'll implement BoW properly in **04-Bag-of-Words**.

---
## 🔗 6. N-Grams

An **N-gram** is a sequence of **N consecutive tokens**.

### Unigram (N=1)

```text
"I love Python"
→ ["I", "love", "Python"]
```

### Bigram (N=2)

```text
→ ["I love", "love Python"]
```

### Trigram (N=3)

```text
→ ["I love Python"]
```

### Why N-grams matter

`"not good"` as a bigram is more informative than treating `not` and `good` **independently**.

N-grams let classical models capture **some** word-order information.

In [ ]:
# N-gram extraction
def ngrams(tokens, n):
    return [' '.join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

text = "not good but not terrible either"
tokens = text.split()

print('Tokens  :', tokens)
print('Unigram :', ngrams(tokens, 1))
print('Bigram  :', ngrams(tokens, 2))
print('Trigram :', ngrams(tokens, 3))
print()
print('Notice how the bigram "not good" preserves the negation signal.')

---
## ⚖️ 7. TF-IDF (Overview)

**TF-IDF = Term Frequency × Inverse Document Frequency**

It weighs each word by:

1. How often it appears in a document → **TF**
2. How rare it is across all documents → **IDF**

### Simplified formula

```text
TF-IDF = TF × IDF

IDF = log(N / df)
```

where:

- `N` = total number of documents
- `df` = number of documents containing the term

### Intuition

| Situation | TF-IDF score |
| --- | --- |
| Common everywhere ("the") | low |
| Frequent in one doc, rare elsewhere ("python") | high |

> Full details in **05-TF-IDF**.

In [ ]:
# Tiny TF-IDF computation (no sklearn) to build intuition
import math

def compute_tf(doc):
    tokens = doc.lower().split()
    return {w: tokens.count(w) / len(tokens) for w in set(tokens)}

def compute_idf(corpus):
    N = len(corpus)
    df = {}
    for doc in corpus:
        for w in set(doc.lower().split()):
            df[w] = df.get(w, 0) + 1
    return {w: math.log(N / df[w]) for w in df}

idf = compute_idf(corpus)

print(f'{"Word":<12} {"IDF":>8}')
print('-' * 22)
for w, v in sorted(idf.items(), key=lambda x: x[1], reverse=True)[:8]:
    print(f'{w:<12} {v:>8.3f}')

print()
print('Rare words get higher IDF — this is what makes TF-IDF powerful.')

---
## 🧬 8. Word Embeddings (Overview)

Word embeddings represent words as **dense** numerical vectors.

### One-hot vs Embedding

```text
One-hot:   Python → [0, 0, 0, 1, 0, 0, ...]
Embedding: Python → [0.21, -0.43, 0.71, 0.18, ...]
```

### Properties

- Every value is (usually) **non-zero**
- Vectors are **smaller** (e.g., 50–300 dims) than one-hot
- They capture **semantic relationships**

### Popular models

```text
Word2Vec
GloVe
FastText
```

> Full details in **06-Word-Embeddings**.

---
## ⚖️ 9. Sparse vs Dense Representation

| Feature | Sparse | Dense |
| --- | --- | --- |
| Example | BoW, TF-IDF | Word2Vec, GloVe |
| Many zeros | ✅ Yes | Usually ❌ no |
| Dimensions | Very large | Smaller |
| Semantic info | Limited | Better |
| Memory | Efficient sparse storage | Stores most values |
| Typical use | Classical ML | Similarity / semantic features |

### Example

```text
TF-IDF     → [0, 0, 0, 0.72, 0, 0.15, 0]
Embedding  → [0.21, -0.43, 0.71, 0.18, 0.52]
```

In [ ]:
# Visual comparison: sparse vs dense
sparse_vec = np.array([0, 0, 0, 0.72, 0, 0.15, 0, 0, 0.31])
dense_vec  = np.array([0.21, -0.43, 0.71, 0.18, 0.52, -0.11, 0.34, 0.05, -0.22])

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].bar(range(len(sparse_vec)), sparse_vec)
axes[0].set_title(f'Sparse (TF-IDF-like) — zeros: {(sparse_vec == 0).sum()}/{len(sparse_vec)}')
axes[0].axhline(0, color='k', linewidth=0.5); axes[0].grid(alpha=0.3, axis='y')

axes[1].bar(range(len(dense_vec)), dense_vec)
axes[1].set_title(f'Dense (embedding-like) — zeros: {(dense_vec == 0).sum()}/{len(dense_vec)}')
axes[1].axhline(0, color='k', linewidth=0.5); axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout(); plt.show()

---
## 📄 10. Document Representation

We don't always need to represent **individual words** — we can represent a **whole document** as one vector.

```text
Document
   ↓
Tokens
   ↓
Word Vectors
   ↓
Aggregation
   ↓
Document Vector
```

Simple aggregation:

```text
Average of word embeddings
```

### Useful for

- Document similarity
- Search
- Clustering
- Recommendation
- Classification

In [ ]:
# Fake embeddings to demonstrate averaging word vectors into a document vector
np.random.seed(42)
fake_embed = {w: np.random.randn(5) for w in
              ['i', 'love', 'machine', 'learning', 'python', 'football']}

def doc_vector(doc):
    vecs = [fake_embed[w] for w in doc.lower().split() if w in fake_embed]
    return np.mean(vecs, axis=0) if vecs else np.zeros(5)

for doc in ["I love machine learning", "I love Python", "football"]:
    print(f'{doc:<30} → {doc_vector(doc).round(3)}')

---
## 📐 11. Text Similarity (Cosine)

Once text becomes vectors, we can compare them.

**Cosine similarity** measures the angle between two vectors:

$$\cos(\theta) = \frac{A \cdot B}{\|A\| \, \|B\|}$$

- `1` → identical direction
- `0` → orthogonal (unrelated)
- `-1` → opposite

### Example

```text
"machine learning tutorial"
"machine learning course"     → high similarity

"machine learning tutorial"
"football match today"        → low similarity
```

In [ ]:
# Cosine similarity on bag-of-words vectors
def cosine(a, b):
    a, b = np.asarray(a), np.asarray(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

# Use the BoW matrix we built earlier
docs = [
    "machine learning tutorial",
    "machine learning course",
    "football match today",
]

# Build vocab from corpus + new docs
all_docs = corpus + docs
vocab = build_vocab(all_docs)
vectors = {d: bow(d, vocab) for d in docs}

print('Pairwise cosine similarity:')
for i, a in enumerate(docs):
    for b in docs[i + 1:]:
        sim = cosine(vectors[a], vectors[b])
        print(f'  {a:<30} vs {b:<30} → {sim:.3f}')

---
## 🎯 12. Choosing a Representation

No single representation is always correct.

### BoW

Useful when:
```text
Word occurrence / counts are enough
```

### TF-IDF

Useful for:
```text
Text classification
Search
Keyword importance
Document similarity
```

### Word Embeddings

Useful when:
```text
Semantic relationships
Similarity
Meaning
```
are important.

### Common classical pipeline

```text
Text
 ↓
Preprocessing
 ↓
TF-IDF
 ↓
Logistic Regression
 ↓
Prediction
```

---
## ⚠️ 13. Important Limitations

Classical representations have limits.

### TF-IDF treats words as independent

```text
"The movie is good"
"The movie is not good"
```

A word-based representation may not fully understand how **not** changes the meaning.

### Traditional embeddings give one vector per word

```text
"river bank"
"bank account"
```

The word `bank` gets the **same** vector in both — even though the meaning is different.

> These limitations motivated **neural NLP** and **Transformer-based models**.

---
## 🧠 14. Mental Model

```text
Text
 ↓
Tokens
 ↓
Representation
 ├── One-Hot
 ├── BoW
 ├── N-Grams
 ├── TF-IDF
 └── Embeddings
       ↓
Numerical Vectors
       ↓
ML / NLP Task
```

### 🧭 Golden Rule

> **Text representation converts language into numerical features while trying to preserve information useful for the NLP task.**

---
## 🏋️ 15. Hands-On Exercises

### Exercise 1 — One-Hot
Build one-hot vectors for the vocabulary:
```text
['apple', 'banana', 'cherry', 'date']
```
What is the vector for `cherry`?

### Exercise 2 — BoW by hand
For the corpus:
```python
["red apple", "green apple", "red car"]
```
build the vocabulary and the BoW matrix.

### Exercise 3 — N-grams
Extract unigrams, bigrams, and trigrams from:
```text
"the quick brown fox jumps"
```

### Exercise 4 — TF-IDF intuition
Compute IDF for each word in:
```python
corpus = [
    "the cat sat",
    "the dog sat",
    "the cat ran",
]
```
Which word has the **lowest** IDF? Why?

### Exercise 5 — Cosine similarity
Represent these three docs using BoW and compute cosine similarity between each pair:
```text
"I love coffee"
"I love tea"
"I hate traffic"
```
Which pair is most similar? Why?

### Exercise 6 — Choose a representation
For each task, pick the most suitable classical representation and justify:
- Spam detection
- Sentiment analysis
- Document search
- Product recommendation (semantic)

---
## 🎤 16. Interview Questions

1. **Why convert text to numbers?**  
   ML algorithms operate on numerical data — text must be encoded as vectors.

2. **What is one-hot encoding?**  
   A vector with a single 1 at the index of the word and 0 elsewhere.

3. **What's the main drawback of one-hot encoding?**  
   Very high-dimensional and no semantic information.

4. **What is a sparse vector?**  
   A vector where most entries are zero — as in BoW and TF-IDF.

5. **What does BoW represent?**  
   Word occurrence and counts — word order is ignored.

6. **Why are N-grams useful?**  
   They capture some local word-order information, e.g., `"not good"`.

7. **What does TF-IDF capture?**  
   It weights terms by how frequent they are in a document and how rare they are across the corpus.

8. **What is a dense representation?**  
   A representation where most values are non-zero — e.g., word embeddings.

9. **Why are embeddings better for similarity?**  
   They capture semantic relationships in a continuous space.

10. **What is cosine similarity?**  
    A measure of the angle between two vectors — often used for text similarity.

11. **How do you build a document vector?**  
    Common approaches: TF-IDF vector, average of word embeddings, or a trained document embedding.

12. **What's the biggest limitation of classical representations?**  
    They treat words largely as independent, ignoring context (e.g., negation, polysemy).

---
## 🏁 17. Key Takeaways

### You now understand

- Why text must be converted into numbers
- One-Hot Encoding
- Sparse representations
- Bag of Words (overview)
- N-Grams
- TF-IDF (overview)
- Dense word embeddings (overview)
- Sparse vs Dense
- Word vs document representation
- Text similarity (cosine)
- When to choose each representation
- Classical limitations

### The progression

```text
One-Hot
 ↓
Bag of Words
 ↓
N-Grams
 ↓
TF-IDF
 ↓
Word Embeddings
 ↓
(Later) Contextual Embeddings
```

---

### 🔗 What's Next?

**04 — Bag of Words**

We'll implement BoW **practically** with Python and scikit-learn, inspect sparse matrices, and see when BoW is still a strong baseline.